# PharmaFlow AI — Phase 2 Walkthrough

This notebook demonstrates all Phase 2 components:
1. **Bulk Purchase Optimizer** — LP-based supplier allocation minimising risk-adjusted cost
2. **Inventory Manager** — Safety stock, EOQ, and reorder recommendations
3. **API Integration** — Testing all FastAPI endpoints

---

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.size'] = 11

PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
print(f'Project root: {PROJECT_ROOT}')

---
## 1. Bulk Purchase Optimizer

The LP optimizer solves:
```
minimise:  Σ (price_per_kg + risk_penalty × risk_score) × quantity
subject to:
  demand coverage    ≥ monthly_demand[drug]
  concentration      ≤ 60% from any single supplier
  supplier capacity  ≤ historical_capacity[supplier]
  x[d,s] = 0        if supplier not approved for drug
```

In [ ]:
from src.optimization.purchase_optimizer import PurchaseOptimizer

# Run with default demand (1 month average per drug)
opt = PurchaseOptimizer(risk_penalty_weight=0.8, concentration_limit=0.60)
result = opt.optimize()

print('=' * 55)
print('PURCHASE OPTIMIZATION RESULTS')
print('=' * 55)
print(f"Solver:              {result['solver']}")
print(f"Drugs covered:       {result['num_drugs']}")
print(f"Suppliers used:      {result['num_suppliers_used']}")
print(f"Raw cost:            ${result['total_raw_cost']:>12,.2f}")
print(f"Risk penalty:        ${result['total_risk_penalty']:>12,.2f}")
print(f"Effective cost:      ${result['total_effective_cost']:>12,.2f}")

In [ ]:
# View the full allocation table
alloc = result['allocation_df']
alloc[['drug_name','supplier_name','supplier_country','quantity_kg',
       'price_per_kg','risk_score','raw_cost_usd','pct_of_demand']].sort_values('drug_name')

In [ ]:
# Visualise: allocation breakdown by drug
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Stacked bar: allocation per drug coloured by supplier
pivot = alloc.pivot_table(index='drug_name', columns='supplier_name',
                           values='quantity_kg', aggfunc='sum').fillna(0)
pivot.plot(kind='barh', stacked=True, ax=axes[0],
           colormap='tab20', legend=False)
axes[0].set_title('Purchase Allocation by Drug & Supplier', fontweight='bold')
axes[0].set_xlabel('Quantity (kg)')
axes[0].set_ylabel('')

# Scatter: risk score vs raw cost per line item
scatter = axes[1].scatter(
    alloc['risk_score'], alloc['raw_cost_usd'],
    c=alloc['quantity_kg'], cmap='YlOrRd',
    s=alloc['quantity_kg'] / alloc['quantity_kg'].max() * 400 + 30,
    alpha=0.7, edgecolors='black', linewidth=0.5
)
plt.colorbar(scatter, ax=axes[1], label='Quantity (kg)')
axes[1].set_xlabel('Supplier Risk Score')
axes[1].set_ylabel('Raw Cost (USD)')
axes[1].set_title('Risk vs Cost per Allocation Line\n(size = quantity)', fontweight='bold')
axes[1].axvline(x=45, color='orange', linestyle='--', alpha=0.5, label='Moderate threshold')
axes[1].axvline(x=65, color='red', linestyle='--', alpha=0.5, label='High threshold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Per-drug summary
summary = result['summary_df']
print('Per-Drug Summary:')
summary[['drug_name','num_suppliers','total_qty_kg','demand_kg','coverage_pct',
          'avg_risk_score','total_raw_cost','total_effective_cost']].sort_values('avg_risk_score', ascending=False)

In [ ]:
# Compare: naive (cheapest only) vs risk-adjusted optimizer
risk_scores = pd.read_csv(os.path.join(PROCESSED_DIR, 'supplier_risk_scores.csv'))
forecasts = pd.read_csv(os.path.join(PROCESSED_DIR, 'price_forecasts.csv'))
forecasts['ds'] = pd.to_datetime(forecasts['ds'])
latest_price = forecasts.sort_values('ds').groupby('drug_id')['yhat'].last()

# Naive: just pick cheapest supplier per drug
import json
drugs_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'synthetic', 'drugs.csv'))
naive_cost = 0
naive_risk_total = 0
for _, drug in drugs_df.iterrows():
    d = drug['id']
    price = float(latest_price.get(d, drug['base_price_per_kg']))
    demand = opt.demand_kg.get(d, 0)
    # Naive: just take first (cheapest) approved supplier
    approved = json.loads(drug.get('approved_suppliers', '[]'))
    if approved:
        cheapest_risk = risk_scores[risk_scores['supplier_id'] == approved[0]]['risk_score']
        risk = float(cheapest_risk.iloc[0]) if not cheapest_risk.empty else 50
        naive_cost += demand * price
        naive_risk_total += demand * opt.risk_penalty_weight * (risk / 100) * 100

opt_raw = result['total_raw_cost']
opt_penalty = result['total_risk_penalty']
opt_eff = result['total_effective_cost']

labels = ['Naive\n(First Supplier)', 'Optimised\n(LP Risk-Adjusted)']
raw_costs = [naive_cost, opt_raw]
risk_penalties = [naive_risk_total, opt_penalty]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(labels))
w = 0.35
bars1 = ax.bar(x - w/2, raw_costs, w, label='Raw Material Cost', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + w/2, risk_penalties, w, label='Risk Penalty', color='#e74c3c', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('USD')
ax.set_title('Naive vs Optimised Purchasing — Cost Breakdown', fontweight='bold', fontsize=13)
ax.legend()

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h * 1.01,
            f'${h:,.0f}', ha='center', va='bottom', fontsize=9)

saving = (naive_cost + naive_risk_total) - opt_eff
ax.text(0.5, 0.95, f'Total effective cost saving: ${saving:,.0f}',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=12, color='green', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))

plt.tight_layout()
plt.show()

---
## 2. Inventory Management

Key formulas:
- **Safety Stock** = Z × σ_demand_daily × √(lead_time_days)  
- **Reorder Point** = avg_daily_demand × lead_time + safety_stock  
- **EOQ** = √(2 × annual_demand × ordering_cost / holding_cost)  
- **Days Cover** = current_stock / avg_daily_demand

In [ ]:
from src.optimization.inventory_manager import InventoryManager

mgr = InventoryManager(service_level=0.95)
inv_df = mgr.compute_recommendations()
s = mgr.summary(inv_df)

print('INVENTORY MANAGEMENT SUMMARY')
print('=' * 45)
print(f"Total drugs:       {s['total_drugs']}")
print(f"⛔ REORDER NOW:    {s['reorder_now']}")
print(f"⚠️  REORDER SOON:   {s['reorder_soon']}")
print(f"✅ ADEQUATE:       {s['adequate']}")
print(f"📦 EXCESS:         {s['excess']}")
print(f"Total stock value: ${s['total_stock_value_usd']:,.2f}")
print(f"Avg days cover:    {s['avg_days_cover']} days")
if s['critical_reorders']:
    print(f"\n🚨 Critical: {s['critical_reorders']}")

In [ ]:
# Full recommendations table
inv_df[['drug_name','criticality','current_stock_kg','reorder_point_kg',
         'safety_stock_kg','eoq_kg','days_cover','action','urgency_score']].sort_values('urgency_score', ascending=False)

In [ ]:
# Inventory dashboard visualisation
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Stock vs Reorder Point
inv_sorted = inv_df.sort_values('urgency_score', ascending=True)
colors_action = {'REORDER_NOW': '#e74c3c', 'REORDER_SOON': '#f39c12',
                  'ADEQUATE': '#2ecc71', 'EXCESS': '#3498db'}
bar_colors = [colors_action.get(a, '#95a5a6') for a in inv_sorted['action']]

axes[0, 0].barh(inv_sorted['drug_name'], inv_sorted['current_stock_kg'],
                color=bar_colors, alpha=0.8, label='Current Stock')
axes[0, 0].scatter(inv_sorted['reorder_point_kg'], range(len(inv_sorted)),
                   color='red', marker='|', s=200, linewidth=2, zorder=5, label='Reorder Point')
axes[0, 0].set_xlabel('Quantity (kg)')
axes[0, 0].set_title('Current Stock vs Reorder Point\n(colour = action)', fontweight='bold')
patches = [mpatches.Patch(color=v, label=k) for k, v in colors_action.items()]
axes[0, 0].legend(handles=patches, fontsize=8, loc='lower right')

# 2. Days Cover
axes[0, 1].barh(inv_sorted['drug_name'], inv_sorted['days_cover'], color=bar_colors, alpha=0.8)
axes[0, 1].axvline(x=30, color='red', linestyle='--', alpha=0.5, label='30-day minimum')
axes[0, 1].axvline(x=60, color='green', linestyle='--', alpha=0.5, label='60-day target')
axes[0, 1].set_xlabel('Days of Cover')
axes[0, 1].set_title('Days of Stock Cover', fontweight='bold')
axes[0, 1].legend(fontsize=9)

# 3. Action distribution pie
action_counts = inv_df['action'].value_counts()
pie_colors = [colors_action.get(a, '#95a5a6') for a in action_counts.index]
axes[1, 0].pie(action_counts.values, labels=action_counts.index,
               colors=pie_colors, autopct='%1.0f%%', startangle=90,
               textprops={'fontsize': 11})
axes[1, 0].set_title('Inventory Status Distribution', fontweight='bold')

# 4. Urgency vs Days Cover scatter
scatter_colors = [colors_action.get(a, '#95a5a6') for a in inv_df['action']]
axes[1, 1].scatter(inv_df['days_cover'], inv_df['urgency_score'],
                   c=scatter_colors, s=150, alpha=0.7, edgecolors='black', linewidth=0.5)
for _, row in inv_df.iterrows():
    axes[1, 1].annotate(row['drug_name'].split()[0],
                        (row['days_cover'], row['urgency_score']),
                        fontsize=7, ha='center', va='bottom')
axes[1, 1].set_xlabel('Days of Cover')
axes[1, 1].set_ylabel('Urgency Score (0-100)')
axes[1, 1].set_title('Urgency vs Days Cover', fontweight='bold')
axes[1, 1].axhline(y=50, color='orange', linestyle='--', alpha=0.4)
axes[1, 1].axhline(y=80, color='red', linestyle='--', alpha=0.4)

plt.suptitle('PharmaFlow AI — Inventory Management Dashboard', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 3. FastAPI Endpoint Tour

Start the API first:
```bash
uvicorn src.api.main:app --reload --port 8000
```

Then run the cells below to test all endpoints.

In [ ]:
import requests
BASE = 'http://localhost:8000'

# Health check
r = requests.get(f'{BASE}/health')
print('GET /health')
print(r.json())

In [ ]:
# List drugs
r = requests.get(f'{BASE}/drugs')
drugs_api = r.json()
print(f'GET /drugs → {len(drugs_api)} drugs')
pd.DataFrame(drugs_api)

In [ ]:
# List suppliers with risk
r = requests.get(f'{BASE}/suppliers')
sups_api = r.json()
print(f'GET /suppliers → {len(sups_api)} suppliers')
pd.DataFrame(sups_api).sort_values('risk_score', ascending=False).head(10)

In [ ]:
# Price forecast for DRG001
r = requests.post(f'{BASE}/forecast/price', json={'drug_id': 'DRG001', 'weeks_ahead': 8})
fc = r.json()
print(f"POST /forecast/price — {fc['drug_name']} (MAPE: {fc['mape_pct']}%)")
fc_df = pd.DataFrame(fc['forecast'])
fc_df.plot(x='date', y='predicted_price', figsize=(12, 4),
           title=f"Price Forecast: {fc['drug_name']}", marker='o')
plt.tight_layout(); plt.show()

In [ ]:
# Supplier risk for SUP001
r = requests.post(f'{BASE}/risk/supplier', json={'supplier_id': 'SUP001'})
print('POST /risk/supplier')
import json
print(json.dumps(r.json(), indent=2))

In [ ]:
# Quality anomaly check — normal batch
normal = {
    'supplier_id': 'SUP001', 'drug_id': 'DRG001',
    'purity_pct': 97.5, 'contamination_ppm': 1.2,
    'moisture_pct': 0.3, 'particle_size_d90': 115.0
}
r = requests.post(f'{BASE}/anomaly/quality', json=normal)
print('POST /anomaly/quality (NORMAL batch)')
print(json.dumps(r.json(), indent=2))

In [ ]:
# Quality anomaly check — suspicious batch
anomalous = {
    'supplier_id': 'SUP001', 'drug_id': 'DRG001',
    'purity_pct': 88.5, 'contamination_ppm': 8.9,
    'moisture_pct': 2.8, 'particle_size_d90': 310.0
}
r = requests.post(f'{BASE}/anomaly/quality', json=anomalous)
print('POST /anomaly/quality (SUSPICIOUS batch)')
print(json.dumps(r.json(), indent=2))

In [ ]:
# Purchase optimizer via API (custom demand)
custom_demand = {
    'demand': [
        {'drug_id': 'DRG001', 'quantity_kg': 5000},
        {'drug_id': 'DRG002', 'quantity_kg': 3000},
        {'drug_id': 'DRG005', 'quantity_kg': 8000},
    ],
    'risk_penalty_weight': 1.2,
    'concentration_limit': 0.50,
}
r = requests.post(f'{BASE}/optimize/purchase', json=custom_demand)
opt_result = r.json()
print(f"POST /optimize/purchase")
print(f"  Solver: {opt_result['solver']}")
print(f"  Effective cost: ${opt_result['total_effective_cost_usd']:,.2f}")
print(f"  Suppliers: {opt_result['num_suppliers_used']}")
pd.DataFrame(opt_result['allocation'])[['drug_name','supplier_name','quantity_kg','risk_score','total_effective_cost']]

In [ ]:
# Inventory recommendations via API
r = requests.post(f'{BASE}/optimize/inventory', json={'service_level': 0.95})
inv_result = r.json()
print('POST /optimize/inventory')
print(f"  Reorder NOW:  {inv_result['reorder_now']}")
print(f"  Reorder SOON: {inv_result['reorder_soon']}")
print(f"  Stock value: ${inv_result['total_stock_value_usd']:,.2f}")
pd.DataFrame(inv_result['recommendations'])[['drug_name','days_cover','action','urgency_score']].head(10)

In [ ]:
# Dashboard summary
r = requests.get(f'{BASE}/dashboard/summary')
dash = r.json()
print('GET /dashboard/summary')
print(f"  Total drugs:      {dash['total_drugs']}")
print(f"  Total suppliers:  {dash['total_suppliers']}")
print(f"  Critical:         {dash['critical_suppliers']}")
print(f"  High risk:        {dash['high_risk_suppliers']}")
print(f"  Needing reorder:  {dash['drugs_needing_reorder']}")
print(f"  Avg MAPE:         {dash['avg_price_forecast_mape_pct']}%")
print(f"  Anomalies found:  {dash['total_anomalies_detected']}")
print(f"\nTop risk suppliers:")
for s in dash['top_risk_suppliers']:
    print(f"  {s['supplier_name']:<25} score={s['risk_score']:.1f}  tier={s['risk_tier']}")
print(f"\nPrice alerts: {len(dash['price_alerts'])}")
for a in dash['price_alerts'][:5]:
    print(f"  {a['drug_name']}: ${a['base_price']} → ${a['forecast_price']} (+{a['pct_increase']}%)")

---
## Summary

| Component | Key Output | Result |
|-----------|-----------|--------|
| **Purchase Optimizer** | Solver | LP (PuLP CBC) |
| | Risk-adjusted cost saving vs naive | See chart |
| | Concentration limit enforced | ≤60% from any one supplier |
| **Inventory Manager** | Drugs tracked | 18 |
| | Service level | 95% (Z=1.645) |
| | Formulas | Safety stock, ROP, EOQ |
| **FastAPI** | Endpoints | 9 across 3 tag groups |
| | Docs | http://localhost:8000/docs |

**Next: Phase 3** — Shortage Prediction + NLP Geopolitical Intelligence layer